# Analysis of Minor League Challenge Systems

In this file we will analyze the data we prepared in `data_prep.ipynb`.

## Questions to Pursue

1) What types of pitches and pitch locations are challenged most frequently?

2) How often are challenges successful?

3) Which batters / teams challenge most frequently?

4) Who uses their challenges most / least effectively? Could incorporate run values here

5) Can we create a challenge 'player card' visualization as a proof of concept?

## Current Blockers

1) It turns out that sz_top and sz_bottom don't align perfectly with the top and bottom of the strike zone. It would be helpful to definitiely determine whether a challenge will be successful to analyze 'missed opportunities'.

## Next Steps

1) Begin analysis with the questions above as a guide.

2) Look into using the zone column to determine whether a pitch is in the strike zone.

# Import Libraries

In [1]:
import numpy as np
import polars as pl

# Load Data

In [2]:
pbp = pl.read_csv('../data/export_data/aaa_pbp.csv')

In [3]:
pbp.head()

pitch_id,play_start_datetime,play_end_datetime,pitch_type,pitch_name,game_date,release_speed,release_pos_x,release_pos_y,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,…,woba_denom,babip_value,iso_value,launch_speed_angle,at_bat_number,pitch_number,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,game_month,game_day,game_year,league_id,league_name,league_level_id,league_level_name,away_team_org_id,away_team_org_name,home_team_org_id,home_team_org_name,game_id,plate_appearance_id,lag_balls,lag_strikes,swing,challenge,challenge_successful
i64,str,str,str,str,str,f64,f64,f64,f64,str,i64,i64,str,str,f64,str,str,str,i64,str,str,str,str,str,str,str,f64,str,i64,i64,f64,f64,f64,f64,f64,f64,…,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,f64,str,str,i64,i64,i64,i64,str,i64,str,i64,str,i64,str,i64,i64,i64,i64,str,i64,str
0,"""2024-06-28T01:53:58.280""","""2024-06-28 01:54:02.789""","""CH""","""Changeup""","""2024-06-27""",86.6,-1.504636,50.000499,5.854567,"""Peter Lambert""",519303,663567,null,"""Elliot Soto strikes out on a f…",229.0,null,null,null,6,"""Elliot Soto strikes out on a f…","""R""","""R""","""R""","""ABQ""","""SL""","""C""",null,null,1,1,-6.941855,2.973652,0.753154,2.103768,null,null,…,null,null,null,null,34,2,6,1,1,6,1,6,1,6,null,null,229.0,null,null,6,27,2024,112,"""Pacific Coast League""",11,"""Triple-A""",108,"""Los Angeles Angels""",115,"""Colorado Rockies""",1036,22017,1,0,"""take""",0,"""no challenge"""
1,"""2024-09-18T00:19:15.010""","""2024-09-18 00:19:25.647""","""FF""","""Four-Seam Fastball""","""2024-09-17""",88.1,3.232269,50.004946,5.669619,"""Mason Fluharty""",666211,689254,null,"""Taylor Trammell called out on …",169.0,null,null,null,4,"""Taylor Trammell called out on …","""R""","""L""","""L""","""BUF""","""SWB""","""B""",null,null,1,0,-2.407521,4.213267,-0.760974,2.705338,null,null,…,null,null,null,null,58,1,3,3,3,3,3,3,3,3,null,null,169.0,null,null,9,17,2024,117,"""International League""",11,"""Triple-A""",147,"""New York Yankees""",141,"""Toronto Blue Jays""",807,22994,0,0,"""take""",0,"""no challenge"""
2,"""2024-07-25T03:06:15.107""","""2024-07-25 03:06:19.295""","""SL""","""Slider""","""2024-07-24""",85.0,0.673434,50.003952,5.825734,"""Blake Taylor""",680862,642130,null,"""Willie MacIver strikes out on …",288.0,null,null,null,13,"""Willie MacIver strikes out on …","""R""","""R""","""L""","""ABQ""","""RR""","""B""",null,null,2,2,-1.537159,-0.80345,-1.014508,0.475927,null,null,…,null,null,null,null,73,7,4,7,4,7,7,4,4,7,null,null,288.0,null,null,7,24,2024,112,"""Pacific Coast League""",11,"""Triple-A""",140,"""Texas Rangers""",115,"""Colorado Rockies""",943,66672,1,2,"""take""",0,"""no challenge"""
3,"""2024-08-03T22:40:47.492""","""2024-08-03 22:40:51.501""","""FF""","""Four-Seam Fastball""","""2024-08-03""",95.2,-0.00569,50.002266,6.428304,"""Braydon Fisher""",681508,680755,null,"""Mickey Gasper walks. Triston…",203.0,null,null,null,11,"""Mickey Gasper walks. Triston…","""R""","""L""","""R""","""WOR""","""BUF""","""B""",null,null,2,2,-3.876377,10.492538,-0.363394,4.554971,805367.0,671213.0,…,null,null,null,null,71,4,9,5,9,5,5,9,9,5,null,null,203.0,null,null,8,3,2024,117,"""International League""",11,"""Triple-A""",141,"""Toronto Blue Jays""",111,"""Boston Red Sox""",615,24923,1,2,"""take""",0,"""no challenge"""
4,"""2024-08-16T23:05:44.932""","""2024-08-16 23:05:51.595""","""CU""","""Curveball""","""2024-08-16""",74.2,-1.255317,50.000262,5.939749,"""Carlos Rodriguez""",682927,692230,null,"""Ronny Simon grounds out, first…",47.0,null,null,null,13,"""Ronny Simon grounds out, first…","""R""","""L""","""R""","""DUR""","""NAS""","""

In [4]:
# add a column to identify the team that challenged (empty if no challenge)
pbp = pbp.with_columns([
    pl.when(
        (pl.col('challenge') == 1) &
        (pl.col('des').str.contains('challenged'))
        )
    .then(
        pl.col('des')
        .str.split('challenged')
        .list.first()
        .str.strip_chars()
    )
    .otherwise(pl.lit(""))
    .alias('challenge_team'),

    pl.when(
        (
            (pl.col('challenge_successful') == 'unsuccessful') &
            (pl.col('balls') > pl.col('lag_balls'))
        ) |
        (
            (pl.col('challenge_successful') == 'successful') &
            (pl.col('balls') == pl.col('lag_balls'))
        )
    )
    .then(pl.lit('ball'))
    .when(
        (
            (pl.col('challenge_successful') == 'unsuccessful') &
            (pl.col('strikes') > pl.col('lag_strikes'))
        ) |
        (
            (pl.col('challenge_successful') == 'successful') &
            (pl.col('strikes') == pl.col('lag_strikes'))
        )
    )
    .then(pl.lit('strike'))
    .otherwise(pl.lit(''))
    .alias('original_call_challenge'),
    
    pl.when(
        (
            (pl.col('challenge_successful') == 'successful') &
            (pl.col('strikes') > pl.col('lag_strikes'))
        ) |
        (
            (pl.col('challenge_successful') == 'unsuccessful') &
            (pl.col('strikes') == pl.col('lag_strikes'))
        )
    )
    .then(pl.lit('Pitching'))
    .when(
        (
            (pl.col('challenge_successful') == 'successful') &
            (pl.col('balls') > pl.col('lag_balls'))
        ) |
        (
            (pl.col('challenge_successful') == 'unsuccessful') &
            (pl.col('balls') == pl.col('lag_balls'))
        )
    )
    .then(pl.lit('Batting'))
    .otherwise(pl.lit(''))
    .alias('challenge_side')
]
)

# Classify Pitch as In or Outside of Strikezone

**Horizontal limits:** x is centered at 0 and measured in feet. Home plate is 17 inches wide, and the strike zone's horizontal limits include home plate plus the diameter of the baseball (2.94 inches) = 19.94 inches = 1.66 feet. Thus, the limits of the strike zone are +- 0.83.

**Vertical limits:** Right now we're using sz_top and sz_bot +/- the diameter of the ball, but as we'll see below it's not entirely consistent with ABS results.

In [5]:
ball_types = ['B', '*B', 'P']
pbp = (
    pbp
        .with_columns(
            pl.when(
                (np.abs(pl.col('plate_x')) < 0.83) &
                (pl.col('plate_z') >= (pl.col('sz_bot') - (2.94 / 12))) &
                (pl.col('plate_z') <= (pl.col('sz_top') + (2.94 / 12)))
            )
            .then(1)
            .otherwise(0)
            .alias('in_strike_zone')
            )
        .with_columns(
            # called a ball, but in the strike zone
            pl.when(
                (
                    (pl.col('type').is_in(ball_types)) &
                    (pl.col('in_strike_zone') == 1)
                ) |
                # called a strike, but not in the strike zone
                (
                    (pl.col('swing') == 'take') &
                    (pl.col('strikes') > pl.col('lag_strikes')) &
                    (pl.col('in_strike_zone') == 0)
                )
            )
            .then(1)
            .otherwise(0)
            .alias('missed_call')
        )
)

# Visualize Normalized Location of Pitches

- Convert height of pitch into percentile of bottom to top of strike zone for consistent plotting

formula: 100 * (pitch height - bottom of zone) / (top of zone - bottom of zone)

In [6]:
delta = 2.94 / 12
pbp = pbp.with_columns(
    (
        100 * (
            pl.col('plate_z') - (pl.col('sz_bot') - delta)
        ) / (
            (pl.col('sz_top') + delta) - (pl.col('sz_bot') - delta)
        )
    )
    # set upper and lower bounds to prevent skewing of graph
    .clip(-100, 200)
    .alias('percentile_z')
)

## Challenges -- Result Strike

In [7]:
(
    pbp
        .filter(
            (pl.col('challenge') == 1) &
            (pl.col('strikes') > pl.col('lag_strikes'))
        )
        .group_by(pl.col('strikes'))
        .len()
)

strikes,len
i64,u32
3,623
1,1
2,1


In [8]:
challenge_s = (
    pbp
        .filter(
            (pl.col('challenge') == 1) &
            (pl.col('strikes') == 3)
        )
)
challenge_s = challenge_s.with_columns(
    pl.when(pl.col('challenge_successful') == 'successful')
    .then(pl.lit('green'))
    .otherwise(pl.lit('red'))
    .alias('challenge_color')
)

In [9]:
import plotly.graph_objects as go

tooltip = challenge_s.select([
    'challenge_team',
    'challenge_side',
    'plate_x',
    'percentile_z',
    'original_call_challenge',
    'challenge_successful'
]).to_numpy()
fig = go.Figure()
# strike zone
fig.add_shape(
    type="rect",
    x0=-0.83, x1=0.83,
    y0=0, y1=100,
    fillcolor="rgba(173, 216, 230, 0.3)",
    line=dict(color="black", width=2)
)
# scatter plot of pitches
fig.add_trace(go.Scatter(
    x=challenge_s["plate_x"],
    y=challenge_s["percentile_z"],
    mode='markers',
    marker=dict(
        color=challenge_s['challenge_color'],
        colorscale = 'RdBu',
        line = dict(width = 0),
        size= 15
        ),
    name='Pitches',
    showlegend = False,
    customdata = tooltip,
    hovertemplate = (
        "Challenge Team: %{customdata[0]}<br>" +
        "Challenge Side: %{customdata[1]}<br>" +
        "Horizontal Location: %{customdata[2]:.2f}<br>" +
        "Vertical Location (%ile): %{customdata[3]:.1f}<br>" +
        "Initial Call: %{customdata[4]}<br>" +
        "Challenge: %{customdata[5]}<extra></extra>"
    )
))
# legend for successful challenges
fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='markers',
    marker=dict(size=15, color='green'),
    name='Successful'
))
# legend for unsuccessful challenges
fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='markers',
    marker=dict(size=15, color='red'),
    name='Unsuccessful'
))
# formatting
fig.update_layout(
    width=500,
    height=700,
    xaxis=dict(
        showgrid=False,
        zeroline=False,
        visible=False,
        range=[-1.1, 1.1]
    ),
    yaxis=dict(
        showgrid=False,
        zeroline=False,
        visible=False,
        range=[-20, 120]
    ),
    plot_bgcolor='white',
    margin=dict(l=0, r=0, t=30, b=0),
    title = dict(
        text = "Locations of Challenges -- Result = Strike",
        font = dict(family = "Arial", size = 24, color = "black"),
        x = 0.5,
        y = 0.96,
        xanchor = 'center'
    ),
    legend = dict(
        y = 0.95,
        x = 0.64
    )
)

fig.show()


## Challenges Resulting in a Ball

In [10]:
challenge_b = (
    pbp
        .filter(
            (pl.col('challenge') == 1) &
            (pl.col('balls') == 4)
        )
        .with_columns(
            pl.when(pl.col('challenge_successful') == 'successful')
            .then(pl.lit('green'))
            .otherwise(pl.lit('red'))
            .alias('challenge_color')
        )
)

In [11]:
import plotly.graph_objects as go

tooltip = challenge_b.select([
    'challenge_team',
    'challenge_side',
    'plate_x',
    'percentile_z',
    'original_call_challenge',
    'challenge_successful'
]).to_numpy()
fig = go.Figure()
# strike zone
fig.add_shape(
    type="rect",
    x0=-0.83, x1=0.83,
    y0=0, y1=100,
    fillcolor="rgba(173, 216, 230, 0.3)",
    line=dict(color="black", width=2)
)
# scatter plot of pitches
fig.add_trace(go.Scatter(
    x=challenge_b["plate_x"],
    y=challenge_b["percentile_z"],
    mode='markers',
    marker=dict(
        color=challenge_b['challenge_color'],
        colorscale = 'RdBu',
        line = dict(width = 0),
        size= 15
        ),
    name='Pitches',
    showlegend = False,
    customdata = tooltip,
    hovertemplate = (
        "Challenge Team: %{customdata[0]}<br>" +
        "Challenge Side: %{customdata[1]}<br>" +
        "Horizontal Location: %{customdata[2]:.2f}<br>" +
        "Vertical Location (%ile): %{customdata[3]:.1f}<br>" +
        "Initial Call: %{customdata[4]}<br>" +
        "Challenge: %{customdata[5]}<extra></extra>"
    )
))
# legend for successful challenges
fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='markers',
    marker=dict(size=15, color='green'),
    name='Successful'
))
# legend for unsuccessful challenges
fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='markers',
    marker=dict(size=15, color='red'),
    name='Unsuccessful'
))
# formatting
fig.update_layout(
    width=500,
    height=700,
    xaxis=dict(
        showgrid=False,
        zeroline=False,
        visible=False,
        range=[-1.2, 1.2]
    ),
    yaxis=dict(
        showgrid=False,
        zeroline=False,
        visible=False,
        range=[-40, 140]
    ),
    plot_bgcolor='white',
    margin=dict(l=0, r=0, t=30, b=0),
    title = dict(
        text = "Locations of Challenges -- Result = Ball",
        font = dict(family = "Arial", size = 24, color = "black"),
        x = 0.5,
        y = 0.96,
        xanchor = 'center'
    ),
    legend = dict(
        y = 0.95,
        x = 0.64
    )
)

fig.show()


# Table of Challenge Success Rates

1) team

2) pitching: num challenges and success rate

3) batting: num challenges and success rate

In [12]:
def success_rate(side: str) -> pl.Expr:
    return (
        pl.when(pl.col("challenge_side") == side)
        .then(pl.col("challenge_successful_binary"))
        .otherwise(None)
        .mean()
        .alias(f"{side.lower()}_success_rate")
    )
team_metrics = (
    pbp
        .with_columns(
            pl.when(pl.col('challenge_successful') == 'successful')
            .then(1)
            .otherwise(0)
            .alias('challenge_successful_binary')
        )
        .filter(pl.col('challenge') == 1)
        .group_by('challenge_team')
        .agg(
            (pl.col('challenge_side') == 'Pitching').sum().alias("num_pitching_challenges"),
            (pl.col('challenge_side') == 'Batting').sum().alias("num_batting_challenges"),
            success_rate('Batting'),
            success_rate('Pitching')
        )
)

In [13]:
from great_tables import GT
import polars.selectors as cs
(
    GT(
        team_metrics
            .sort('num_batting_challenges', descending = True)
    )
    .cols_move(
        columns = [ 'num_batting_challenges', 'batting_success_rate', 'num_pitching_challenges', 'pitching_success_rate'],
        after = 'challenge_team'
    )
    .tab_header("Team Challenge Metrics")
    .tab_spanner(
        "Pitching",
        columns = ['num_pitching_challenges', 'pitching_success_rate']
    )
    .tab_spanner(
        "Batting",
        columns = ['num_batting_challenges', 'batting_success_rate']
    )
    .cols_label(
        challenge_team = "Team",
        num_pitching_challenges = "#",
        pitching_success_rate = "Win Rate",
        num_batting_challenges = "#",
        batting_success_rate = "Win Rate"
    )
    .fmt_percent(['pitching_success_rate', 'batting_success_rate'], decimals = 1)
    .opt_stylize(1)
)

GT(_tbl_data=shape: (30, 5)
┌────────────────┬────────────────────┬────────────────────┬───────────────────┬───────────────────┐
│ challenge_team ┆ num_pitching_chall ┆ num_batting_challe ┆ batting_success_r ┆ pitching_success_ │
│ ---            ┆ enges              ┆ nges               ┆ ate               ┆ rate              │
│ str            ┆ ---                ┆ ---                ┆ ---               ┆ ---               │
│                ┆ u32                ┆ u32                ┆ f64               ┆ f64               │
╞════════════════╪════════════════════╪════════════════════╪═══════════════════╪═══════════════════╡
│ Space Cowboys  ┆ 7                  ┆ 35                 ┆ 0.171429          ┆ 0.571429          │
│ Isotopes       ┆ 11                 ┆ 31                 ┆ 0.193548          ┆ 0.272727          │
│ River Cats     ┆ 8                  ┆ 29                 ┆ 0.172414          ┆ 0.25              │
│ Bats           ┆ 3                  ┆ 29                 ┆ 0.413793          ┆ 0.333333          │
│ Aviators       ┆ 14                 ┆ 28                 ┆ 0.25              ┆ 0.285714          │
│ …              ┆ …                  ┆ …                  ┆ …                 ┆ …                 │
│ Jumbo Shrimp   ┆ 9                  ┆ 17                 ┆ 0.117647          ┆ 0.444444          │
│ Bulls          ┆ 5                  ┆ 16                 ┆ 0.125             ┆ 0.0               │
│ IronPigs       ┆ 4                  ┆ 16                 ┆ 0.0625            ┆ 0.25              │
│ Indians        ┆ 8                  ┆ 13                 ┆ 0.0               ┆ 0.125             │
│ Stripers       ┆ 6                  ┆ 5                  ┆ 0.2               ┆ 0.166667          │
└────────────────┴────────────────────┴────────────────────┴───────────────────┴───────────────────┘, _body=<great_tables._gt_data.Body object at 0x000001D58E538C20>, _boxhead=Boxhead([ColInfo(var='challenge_team', type=<ColInfoTypeEnum.default: 1>, column_label='Team', column_align='left', column_width=None), ColInfo(var='num_batting_challenges', type=<ColInfoTypeEnum.default: 1>, column_label='#', column_align='right', column_width=None), ColInfo(var='batting_success_rate', type=<ColInfoTypeEnum.default: 1>, column_label='Win Rate', column_align='right', column_width=None), ColInfo(var='num_pitching_challenges', type=<ColInfoTypeEnum.default: 1>, column_label='#', column_align='right', column_width=None), ColInfo(var='pitching_success_rate', type=<ColInfoTypeEnum.default: 1>, column_label='Win Rate', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x000001D58E538980>, _spanners=Spanners([SpannerInfo(spanner_id='Pitching', spanner_level=0, spanner_label='Pitching', spanner_units=None, spanner_pattern=None, vars=['num_pitching_challenges', 'pitching_success_rate'], built=None), SpannerInfo(spanner_id='Batting', spanner_level=0, spanner_label='Batting', spanner_units=None, spanner_pattern=None, vars=['num_batting_challenges', 'batting_success_rate'], built=None)]), _heading=Heading(title='Team Challenge Metrics', subtitle=None, preheader=None), _stubhead=None, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x000001D58E539010>, _formats=[<great_tables._gt_data.FormatInfo object at 0x000001D58E538D70>], _substitutions=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional

Other columns to add to this table:

- RV gained / lost
- Missed challenge opportunities

# Challenges by Team

Goal: write function to show the challenges by team and side (pitching / batting)

Next: missed challenge opportunities

In [14]:
def plot_challenge_results(df, challenge_team, challenge_side):
    """ 
    Plots challenge results for the inputted team and side (batting or pitching).

    Parameters
    ----------
    df: polars.DataFrame
        play-by-play dataframe containing challenge information and pitch locations.
    challenge_team: str
        The name of the challenging team (e.g., "IronPigs")
    challenge_side: str
        Indicates whether the challenging team was "Batting" or "Pitching".

    Returns
    -------
    None
        Displays an interactive Plotly figure with pitch locations and challenge results.
    """
    challenge_df = (
        df
            .filter(
                (pl.col('challenge_team') == challenge_team) &
                (pl.col('challenge_side') == challenge_side))
            .with_columns(
                pl.when(pl.col('challenge_successful') == 'successful')
                .then(pl.lit('green'))
                .otherwise(pl.lit('red'))
                .alias('challenge_color'))
    )
    import plotly.graph_objects as go

    tooltip = challenge_df.select([
        'challenge_team',
        'challenge_side',
        'plate_x',
        'percentile_z',
        'original_call_challenge',
        'challenge_successful'
    ]).to_numpy()
    fig = go.Figure()
    # strike zone
    fig.add_shape(
        type="rect",
        x0=-0.83, x1=0.83,
        y0=0, y1=100,
        fillcolor="rgba(173, 216, 230, 0.3)",
        line=dict(color="black", width=2)
    )
    # scatter plot of pitches
    fig.add_trace(go.Scatter(
        x=challenge_df["plate_x"],
        y=challenge_df["percentile_z"],
        mode='markers',
        marker=dict(
            color=challenge_df['challenge_color'],
            colorscale = 'RdBu',
            line = dict(width = 0),
            size= 15
            ),
        name='Pitches',
        showlegend = False,
        customdata = tooltip,
        hovertemplate = (
            "Challenge Team: %{customdata[0]}<br>" +
            "Challenge Side: %{customdata[1]}<br>" +
            "Horizontal Location: %{customdata[2]:.2f}<br>" +
            "Vertical Location (%ile): %{customdata[3]:.1f}<br>" +
            "Initial Call: %{customdata[4]}<br>" +
            "Challenge: %{customdata[5]}<extra></extra>"
        )
    ))
    # legend for successful challenges
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=15, color='green'),
        name='Successful'
    ))
    # legend for unsuccessful challenges
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=15, color='red'),
        name='Unsuccessful'
    ))
    # formatting
    fig.update_layout(
        width=500,
        height=700,
        xaxis=dict(
            showgrid=False,
            zeroline=False,
            visible=False,
            range=[-1.1, 1.1]
        ),
        yaxis=dict(
            showgrid=False,
            zeroline=False,
            visible=False,
            range=[-20, 120]
        ),
        plot_bgcolor='white',
        margin=dict(l=0, r=0, t=30, b=0),
        title = dict(
            text = f"{challenge_team} {challenge_side} Challenges",
            font = dict(family = "Arial", size = 24, color = "black"),
            x = 0.5,
            y = 0.96,
            xanchor = 'center'
        ),
        legend = dict(
            y = 0.95,
            x = 0.64
        )
    )

    fig.show()

    

In [15]:
plot_challenge_results(pbp, 'IronPigs', 'Pitching')

## Dash App

In [34]:
import dash
from dash import dcc, html, Output, Input
import plotly.graph_objects as go

team_names = pbp['challenge_team'].unique().to_list()

# Initialize Dash app
app = dash.Dash(__name__)

app.layout = html.Div(
    children=[
        html.H1("Challenge Results Dashboard", style={'color': 'white', 'textAlign': 'center'}),

        html.Div([
            html.Label("Select Team:", style = {'color': 'white'}),
            dcc.Dropdown(
                id='team-dropdown',
                options=[{'label': team, 'value': team} for team in team_names],
                value=team_names[0],
                style={'width': '50%'}
            ),
        ], style={'margin': '10px'}),

        html.Div([
            html.Label("Select Side:", style = {'color': 'white'}),
            dcc.Dropdown(
                id='side-dropdown',
                options=[
                    {'label': 'Batting', 'value': 'Batting'},
                    {'label': 'Pitching', 'value': 'Pitching'}
                ],
                value='Batting',
                style={'width': '50%'}
            ),
        ], style={'margin': '10px'}),

        dcc.Graph(id='challenge-graph')
    ]
)

@app.callback(
    Output('challenge-graph', 'figure'),
    Input('team-dropdown', 'value'),
    Input('side-dropdown', 'value')
)
def update_graph(selected_team, selected_side):
    challenge_df = (
        pbp
            .filter(
                (pl.col('challenge_team') == selected_team) & 
                (pl.col('challenge_side') == selected_side))
            .with_columns(
                pl.when(pl.col('challenge_successful') == 'successful')
                .then(pl.lit('green'))
                .otherwise(pl.lit('red'))
                .alias('challenge_color'))
    )

    tooltip = challenge_df.select([
        'challenge_team',
        'challenge_side',
        'plate_x',
        'percentile_z',
        'original_call_challenge',
        'challenge_successful'
    ]).to_numpy()

    fig = go.Figure()

    # strike zone
    fig.add_shape(
        type="rect",
        x0=-0.83, x1=0.83,
        y0=0, y1=100,
        fillcolor="rgba(173, 216, 230, 0.3)",
        line=dict(color="black", width=2)
    )

    # scatter plot of pitches
    fig.add_trace(go.Scatter(
        x=challenge_df["plate_x"],
        y=challenge_df["percentile_z"],
        mode='markers',
        marker=dict(
            color=challenge_df['challenge_color'],
            line=dict(width=0),
            size=15
        ),
        name='Pitches',
        showlegend=False,
        customdata=tooltip,
        hovertemplate=(
            "Challenge Team: %{customdata[0]}<br>" +
            "Challenge Side: %{customdata[1]}<br>" +
            "Horizontal Location: %{customdata[2]:.2f}<br>" +
            "Vertical Location (%ile): %{customdata[3]:.1f}<br>" +
            "Initial Call: %{customdata[4]}<br>" +
            "Challenge: %{customdata[5]}<extra></extra>"
        )
    ))

    # legend for successful challenges
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=15, color='green'),
        name='Successful'
    ))

    # legend for unsuccessful challenges
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=15, color='red'),
        name='Unsuccessful'
    ))

    # formatting
    fig.update_layout(
        width=500,
        height=700,
        xaxis=dict(showgrid=False, zeroline=False, visible=False, range=[-1.1, 1.1]),
        yaxis=dict(showgrid=False, zeroline=False, visible=False, range=[-20, 120]),
        plot_bgcolor='white',
        margin=dict(l=0, r=0, t=30, b=0),
        title=dict(
            text=f"{selected_team} {selected_side} Challenges",
            font=dict(family="Arial", size=24, color="black"),
            x=0.5,
            y=0.96,
            xanchor='center'
        ),
        legend=dict(
            y=0.95,
            x=0.64,
            font=dict(color="black"),
            bgcolor='rgba(0,0,0,0)'
        )
    )
    return fig

if __name__ == '__main__':
    app.run(debug=True)


# Missed Challenge Opportunities

1) called strike (lag strikes > strikes, take), was the third strike of at-bat (otherwise wouldn't have that info), not in zone, no challenge

## Connect org id to team nickname

Still in progress 
- connect team id to mlb org with join of pbp and teams
- join that back in on parent org names and filter to league_name (triple a leagues) to get team_nickname

In [17]:
teams = pl.read_csv('../data/raw_data/2024_teams.csv')
major_league_lookup = teams.select('team_id', 'parent_org_name')
triple_a_teams = teams.filter(pl.col('league_name').is_in(['International League', 'Pacific Coast League'])).select('parent_org_name', 'team_nickname')

In [18]:
team_lookup = (
    teams
        .filter(pl.col('league_name').is_in(['International League', 'Pacific Coast League']))
        .select(
            pl.col('parent_org_name').alias('org_name'),
            pl.col('team_nickname').alias('team_name')
        )
)

In [19]:
pbp = (
    pbp.join(
        team_lookup,
        left_on = 'home_team_org_name',
        right_on = 'org_name')
        .rename({'team_name': 'home_team_name'})
        .join(
            team_lookup,
            left_on = 'away_team_org_name',
            right_on = 'org_name')
        .rename({'team_name': 'away_team_name'})
)

In [20]:
pbp.join(team_lookup, left_on = 'home_team_org_name', right_on = 'org_name').select('home_team', 'team_name')

home_team,team_name
str,str
"""ABQ""","""Isotopes"""
"""BUF""","""Bisons"""
"""ABQ""","""Isotopes"""
"""WOR""","""Red Sox"""
"""DUR""","""Bulls"""
…,…
"""MEM""","""Redbirds"""
"""JAX""","""Jumbo Shrimp"""
"""LV""","""Aviators"""


In [21]:
pbp = pbp.with_columns([
    pl.when(pl.col('inning_top_bot') == 'Top')
    .then(pl.col('home')),
    
    pl.when(
        # pitch was a strike
        (pl.col('strikes') > pl.col('lag_strikes')) &
        # third strike (otherwise wouldn't be able to see challenges)
        (pl.col('strikes') == 3) &
        # no swing
        (pl.col('swing') == 'take') &
        # wasn't in the strike zone (theoretically; using the plate operator heights)
        (pl.col('in_strike_zone') == 0)
    )
    .then(1)
    .otherwise(0)
    .alias('missed_challenge_batter')
]
)

ColumnNotFoundError: home

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'with_columns' <---
DF ["pitch_id", "play_start_datetime", "play_end_datetime", "pitch_type", ...]; PROJECT */119 COLUMNS